In [ ]:
# Gerekli kütüphaneler içe aktarılıyor.
# json: Verileri dosyaya kaydetmek ve dosyadan okumak için kullanılır.
# os: Dosyanın var olup olmadığını kontrol etmek için kullanılır.
# ABC ve abstractmethod: Soyut sınıf oluşturmak için kullanılır.
import json
import os
from abc import ABC, abstractmethod


# SistemUyesi soyut temel sınıftır.
# Öğrenci ve öğretmen sınıfları bu sınıftan miras alır.
class SistemUyesi(ABC):
    # Constructor metodu. Nesne oluşurken ID, ad ve soyad bilgilerini alır.
    def __init__(self, id_no, ad, soyad):
        self._id_no = id_no  # Protected değişken. Encapsulation örneğidir.
        self.ad = ad
        self.soyad = soyad

    # Abstract metod. Alt sınıflarda mutlaka yeniden yazılmalıdır.
    @abstractmethod
    def bilgileri_goster(self):
        pass


# Ogretmen sınıfı SistemUyesi sınıfından miras alır.
# Bu kullanım inheritance yani kalıtım örneğidir.
class Ogretmen(SistemUyesi):
    def __init__(self, id_no, ad, soyad, brans):
        super().__init__(id_no, ad, soyad)
        self.brans = brans

    # Polymorphism örneği: Aynı metod öğretmen için farklı çıktı verir.
    def bilgileri_goster(self):
        return f"Öğretmen ID: {self._id_no} | {self.ad} {self.soyad} | Branş: {self.brans}"


# Ogrenci sınıfı da SistemUyesi sınıfından miras alır.
class Ogrenci(SistemUyesi):
    def __init__(self, id_no, ad, soyad):
        super().__init__(id_no, ad, soyad)

        # Notlar private olarak tutulur.
        # Dışarıdan doğrudan değiştirilemez, bu encapsulation örneğidir.
        self.__notlar = []

    # Private notlar listesine kontrollü erişim sağlar.
    def get_notlar(self):
        return self.__notlar

    # Öğrenciye not ekler.
    # Sadece Not sınıfından oluşturulmuş nesneler eklenebilir.
    def not_ekle(self, not_nesnesi):
        if isinstance(not_nesnesi, Not):
            self.__notlar.append(not_nesnesi)
        else:
            raise ValueError("Eklenecek nesne Not sınıfından olmalıdır.")

    # Polymorphism örneği: Aynı metod öğrenci için farklı çıktı verir.
    def bilgileri_goster(self):
        return f"Öğrenci No: {self._id_no} | Ad Soyad: {self.ad} {self.soyad}"

    # Öğrencinin tüm notlarının ortalamasını hesaplar.
    def ortalama_hesapla(self):
        if not self.__notlar:
            return 0.0

        toplam = sum(n.get_not_degeri() for n in self.__notlar)
        return toplam / len(self.__notlar)


# Ders sınıfı ders kodu, ders adı ve öğretmen bilgisini tutar.
class Ders:
    def __init__(self, ders_kodu, ders_adi, ogretmen):
        self.ders_kodu = ders_kodu
        self.ders_adi = ders_adi
        self.ogretmen = ogretmen


# Not sınıfı öğrencinin bir dersten aldığı notu temsil eder.
class Not:
    def __init__(self, ders, not_degeri):
        self.ders = ders

        # Not değeri 0 ile 100 arasında olmalıdır.
        # Hatalı not girilirse ValueError hatası oluşturulur.
        if 0 <= not_degeri <= 100:
            self.__not_degeri = not_degeri
        else:
            raise ValueError("Not değeri 0 ile 100 arasında olmalıdır.")

    # Private not değerine kontrollü erişim sağlar.
    def get_not_degeri(self):
        return self.__not_degeri


# OkulSistemi projenin ana yönetim sınıfıdır.
# Öğrenci, öğretmen, ders, not, dosya kayıt ve menü işlemleri buradan yönetilir.
class OkulSistemi:
    def __init__(self):
        # Veriler sözlük yapısında tutulur.
        # Böylece öğrenci numarası, öğretmen ID veya ders kodu ile hızlı erişim sağlanır.
        self.ogrenciler = {}
        self.ogretmenler = {}
        self.dersler = {}

    # Sisteme yeni öğretmen ekler.
    def ogretmen_ekle(self, ogretmen_id, ad, soyad, brans):
        if ogretmen_id in self.ogretmenler:
            print("[-] Bu ID ile kayıtlı bir öğretmen zaten var.")
            return False

        self.ogretmenler[ogretmen_id] = Ogretmen(ogretmen_id, ad, soyad, brans)
        print("[+] Öğretmen başarıyla eklendi.")
        return True

    # Sistemdeki öğretmenleri listeler.
    def ogretmenleri_listele(self):
        print("\n--- Öğretmenler ---")

        if not self.ogretmenler:
            print("Kayıtlı öğretmen yok.")
            return

        for ogretmen in self.ogretmenler.values():
            print(ogretmen.bilgileri_goster())

    # Sisteme yeni öğrenci ekler.
    def ogrenci_ekle(self, ogrenci_no, ad, soyad):
        if ogrenci_no in self.ogrenciler:
            print("[-] Bu numaraya sahip bir öğrenci zaten kayıtlı.")
            return False

        self.ogrenciler[ogrenci_no] = Ogrenci(ogrenci_no, ad, soyad)
        print("[+] Öğrenci başarıyla eklendi.")
        return True

    # Sistemdeki öğrencileri listeler.
    def ogrencileri_listele(self):
        print("\n--- Öğrenciler ---")

        if not self.ogrenciler:
            print("Kayıtlı öğrenci yok.")
            return

        for ogrenci in self.ogrenciler.values():
            print(f"{ogrenci.bilgileri_goster()} | Ortalama: {ogrenci.ortalama_hesapla():.2f}")

    # Öğrenci ad ve soyad bilgisini günceller.
    def ogrenci_guncelle(self, ogrenci_no, yeni_ad, yeni_soyad):
        if ogrenci_no not in self.ogrenciler:
            print("[-] Öğrenci bulunamadı.")
            return False

        self.ogrenciler[ogrenci_no].ad = yeni_ad
        self.ogrenciler[ogrenci_no].soyad = yeni_soyad
        print("[+] Öğrenci bilgileri güncellendi.")
        return True

    # Öğrenciyi sistemden siler.
    def ogrenci_sil(self, ogrenci_no):
        if ogrenci_no not in self.ogrenciler:
            print("[-] Öğrenci bulunamadı.")
            return False

        del self.ogrenciler[ogrenci_no]
        print("[+] Öğrenci silindi.")
        return True

    # Sisteme yeni ders ekler.
    # Ders eklenebilmesi için öğretmenin daha önce sisteme eklenmiş olması gerekir.
    def ders_ekle(self, ders_kodu, ders_adi, ogretmen_id):
        if ders_kodu in self.dersler:
            print("[-] Bu kodla kayıtlı bir ders zaten var.")
            return False

        if ogretmen_id not in self.ogretmenler:
            print("[-] Öğretmen bulunamadı. Önce öğretmeni sisteme ekleyin.")
            return False

        ogretmen = self.ogretmenler[ogretmen_id]
        self.dersler[ders_kodu] = Ders(ders_kodu, ders_adi, ogretmen)
        print("[+] Ders başarıyla eklendi.")
        return True

    # Sistemdeki dersleri listeler.
    def dersleri_listele(self):
        print("\n--- Dersler ---")

        if not self.dersler:
            print("Kayıtlı ders yok.")
            return

        for ders in self.dersler.values():
            print(
                f"Kod: {ders.ders_kodu} | Ders: {ders.ders_adi} | "
                f"Öğretmen: {ders.ogretmen.ad} {ders.ogretmen.soyad}"
            )

    # Ders adını ve öğretmenini günceller.
    def ders_guncelle(self, ders_kodu, yeni_ders_adi, yeni_ogretmen_id):
        if ders_kodu not in self.dersler:
            print("[-] Ders bulunamadı.")
            return False

        if yeni_ogretmen_id not in self.ogretmenler:
            print("[-] Öğretmen bulunamadı.")
            return False

        self.dersler[ders_kodu].ders_adi = yeni_ders_adi
        self.dersler[ders_kodu].ogretmen = self.ogretmenler[yeni_ogretmen_id]
        print("[+] Ders bilgileri güncellendi.")
        return True

    # Dersi sistemden siler.
    def ders_sil(self, ders_kodu):
        if ders_kodu not in self.dersler:
            print("[-] Ders bulunamadı.")
            return False

        del self.dersler[ders_kodu]
        print("[+] Ders silindi.")
        return True

    # Öğrenciye seçilen ders için not girer.
    def not_gir(self, ogrenci_no, ders_kodu, not_degeri):
        if ogrenci_no not in self.ogrenciler:
            print("[-] Öğrenci bulunamadı.")
            return False

        if ders_kodu not in self.dersler:
            print("[-] Ders bulunamadı.")
            return False

        try:
            not_nesnesi = Not(self.dersler[ders_kodu], not_degeri)
            self.ogrenciler[ogrenci_no].not_ekle(not_nesnesi)
            print("[+] Not başarıyla girildi.")
            return True
        except ValueError as hata:
            print(f"[-] Hata: {hata}")
            return False

    # Bir öğrencinin bütün notlarını ekrana yazdırır.
    def ogrenci_notlarini_goster(self, ogrenci_no):
        if ogrenci_no not in self.ogrenciler:
            print("[-] Öğrenci bulunamadı.")
            return

        ogrenci = self.ogrenciler[ogrenci_no]
        notlar = ogrenci.get_notlar()

        print(f"\n--- {ogrenci.ad} {ogrenci.soyad} Notları ---")

        if not notlar:
            print("Henüz girilmiş not yok.")
            return

        for not_nesnesi in notlar:
            print(
                f"Ders: {not_nesnesi.ders.ders_adi} "
                f"({not_nesnesi.ders.ders_kodu}) | Not: {not_nesnesi.get_not_degeri()}"
            )

        print(f"Genel Ortalama: {ogrenci.ortalama_hesapla():.2f}")

    # Öğrencinin genel ortalamasını hesaplar.
    def ortalama_hesapla(self, ogrenci_no):
        if ogrenci_no not in self.ogrenciler:
            print("[-] Öğrenci bulunamadı.")
            return None

        ogrenci = self.ogrenciler[ogrenci_no]
        ortalama = ogrenci.ortalama_hesapla()

        print(f"[+] {ogrenci.ad} {ogrenci.soyad} genel ortalaması: {ortalama:.2f}")
        return ortalama

    # Başarı barajının üstünde kalan öğrencileri listeler.
    def basarili_ogrencileri_listele(self, baraj_notu=50.0):
        print(f"\n--- Ortalaması {baraj_notu} ve üzeri olan öğrenciler ---")

        bulundu = False

        for ogrenci_no, ogrenci in self.ogrenciler.items():
            ortalama = ogrenci.ortalama_hesapla()

            if ortalama >= baraj_notu:
                print(f"No: {ogrenci_no} | {ogrenci.ad} {ogrenci.soyad} | Ortalama: {ortalama:.2f}")
                bulundu = True

        if not bulundu:
            print("Bu kritere uyan öğrenci bulunamadı.")

    # Öğrenciyi ad, soyad veya öğrenci numarası ile arar.
    def ogrenci_ara(self, arama_metni):
        print(f"\n--- '{arama_metni}' Arama Sonuçları ---")

        bulundu = False
        arama_metni = arama_metni.lower()

        for ogrenci_no, ogrenci in self.ogrenciler.items():
            if (
                arama_metni in ogrenci.ad.lower()
                or arama_metni in ogrenci.soyad.lower()
                or arama_metni == ogrenci_no.lower()
            ):
                print(f"{ogrenci.bilgileri_goster()} | Ortalama: {ogrenci.ortalama_hesapla():.2f}")
                bulundu = True

        if not bulundu:
            print("Eşleşen öğrenci bulunamadı.")

    # Ders bazlı başarı analizi yapar.
    # Ders ortalaması, en yüksek not, en düşük not, geçen ve kalan öğrenci sayısını gösterir.
    def ders_bazli_basari_analizi(self, ders_kodu):
        if ders_kodu not in self.dersler:
            print("[-] Ders bulunamadı.")
            return

        ders = self.dersler[ders_kodu]
        notlar = []

        for ogrenci in self.ogrenciler.values():
            for not_nesnesi in ogrenci.get_notlar():
                if not_nesnesi.ders.ders_kodu == ders_kodu:
                    notlar.append(not_nesnesi.get_not_degeri())

        print(f"\n--- {ders.ders_adi} Dersi Başarı Analizi ---")

        if not notlar:
            print("Bu derse ait girilmiş not bulunmamaktadır.")
            return

        ortalama = sum(notlar) / len(notlar)
        gecen_sayisi = len([not_degeri for not_degeri in notlar if not_degeri >= 50])
        kalan_sayisi = len(notlar) - gecen_sayisi

        print(f"Ders Kodu: {ders.ders_kodu}")
        print(f"Ders Adı: {ders.ders_adi}")
        print(f"Öğretmen: {ders.ogretmen.ad} {ders.ogretmen.soyad}")
        print(f"Not Sayısı: {len(notlar)}")
        print(f"Ders Ortalaması: {ortalama:.2f}")
        print(f"En Yüksek Not: {max(notlar)}")
        print(f"En Düşük Not: {min(notlar)}")
        print(f"Geçen Öğrenci Sayısı: {gecen_sayisi}")
        print(f"Kalan Öğrenci Sayısı: {kalan_sayisi}")

    # Sistemdeki tüm verileri JSON dosyasına kaydeder.
    def verileri_kaydet(self, dosya_adi="data.json"):
        try:
            data = {
                "ogretmenler": {},
                "dersler": {},
                "ogrenciler": {}
            }

            for ogretmen_id, ogretmen in self.ogretmenler.items():
                data["ogretmenler"][ogretmen_id] = {
                    "ad": ogretmen.ad,
                    "soyad": ogretmen.soyad,
                    "brans": ogretmen.brans
                }

            for ders_kodu, ders in self.dersler.items():
                data["dersler"][ders_kodu] = {
                    "ders_adi": ders.ders_adi,
                    "ogretmen_id": ders.ogretmen._id_no
                }

            for ogrenci_no, ogrenci in self.ogrenciler.items():
                notlar = []

                for not_nesnesi in ogrenci.get_notlar():
                    notlar.append({
                        "ders_kodu": not_nesnesi.ders.ders_kodu,
                        "not_degeri": not_nesnesi.get_not_degeri()
                    })

                data["ogrenciler"][ogrenci_no] = {
                    "ad": ogrenci.ad,
                    "soyad": ogrenci.soyad,
                    "notlar": notlar
                }

            with open(dosya_adi, "w", encoding="utf-8") as dosya:
                json.dump(data, dosya, ensure_ascii=False, indent=4)

            print(f"[+] Veriler '{dosya_adi}' dosyasına kaydedildi.")
            return True

        except Exception as hata:
            print(f"[-] Dosya kaydedilirken hata oluştu: {hata}")
            return False

    # JSON dosyasındaki verileri sisteme geri yükler.
    def verileri_yukle(self, dosya_adi="data.json"):
        if not os.path.exists(dosya_adi):
            print(f"[-] '{dosya_adi}' dosyası bulunamadı.")
            return False

        try:
            with open(dosya_adi, "r", encoding="utf-8") as dosya:
                data = json.load(dosya)

            self.ogretmenler.clear()
            self.dersler.clear()
            self.ogrenciler.clear()

            for ogretmen_id, ogretmen in data.get("ogretmenler", {}).items():
                self.ogretmenler[ogretmen_id] = Ogretmen(
                    ogretmen_id,
                    ogretmen["ad"],
                    ogretmen["soyad"],
                    ogretmen["brans"]
                )

            for ders_kodu, ders in data.get("dersler", {}).items():
                ogretmen_id = ders["ogretmen_id"]

                if ogretmen_id in self.ogretmenler:
                    self.dersler[ders_kodu] = Ders(
                        ders_kodu,
                        ders["ders_adi"],
                        self.ogretmenler[ogretmen_id]
                    )

            for ogrenci_no, ogrenci in data.get("ogrenciler", {}).items():
                self.ogrenciler[ogrenci_no] = Ogrenci(
                    ogrenci_no,
                    ogrenci["ad"],
                    ogrenci["soyad"]
                )

                for not_bilgisi in ogrenci.get("notlar", []):
                    ders_kodu = not_bilgisi["ders_kodu"]

                    if ders_kodu in self.dersler:
                        self.not_gir(
                            ogrenci_no,
                            ders_kodu,
                            not_bilgisi["not_degeri"]
                        )

            print(f"[+] Veriler '{dosya_adi}' dosyasından yüklendi.")
            return True

        except Exception as hata:
            print(f"[-] Dosya yüklenirken hata oluştu: {hata}")
            return False


# Programın ana menüsü burada çalışır.
# Kullanıcı terminalden işlem seçerek sistemi kullanır.
def ana_menu():
    sistem = OkulSistemi()

    # Program ilk açıldığında örnek bir öğretmen eklenir.
    sistem.ogretmen_ekle("T1", "Merve", "Hocam", "Yapay Zeka")

    while True:
        print("\n" + "=" * 50)
        print(" ÖĞRENCİ BİLGİ VE NOT TAKİP SİSTEMİ ")
        print("=" * 50)
        print("1- Öğrenci ekle")
        print("2- Öğrencileri listele")
        print("3- Öğrenci güncelle")
        print("4- Öğrenci sil")
        print("5- Öğretmen ekle")
        print("6- Öğretmenleri listele")
        print("7- Ders ekle")
        print("8- Dersleri listele")
        print("9- Ders güncelle")
        print("10- Ders sil")
        print("11- Not gir")
        print("12- Öğrenci notlarını görüntüle")
        print("13- Ortalama hesapla")
        print("14- Başarılı öğrencileri listele")
        print("15- Öğrenci ara")
        print("16- Ders bazlı başarı analizi")
        print("17- Verileri kaydet")
        print("18- Verileri yükle")
        print("0- Çıkış")
        print("=" * 50)

        try:
            secim = input("Seçiminiz: ").strip()

            if secim == "1":
                no = input("Öğrenci No: ").strip()
                ad = input("Ad: ").strip()
                soyad = input("Soyad: ").strip()

                if no and ad and soyad:
                    sistem.ogrenci_ekle(no, ad, soyad)
                else:
                    print("[-] Alanlar boş bırakılamaz.")

            elif secim == "2":
                sistem.ogrencileri_listele()

            elif secim == "3":
                no = input("Güncellenecek öğrenci no: ").strip()
                ad = input("Yeni ad: ").strip()
                soyad = input("Yeni soyad: ").strip()

                if no and ad and soyad:
                    sistem.ogrenci_guncelle(no, ad, soyad)
                else:
                    print("[-] Alanlar boş bırakılamaz.")

            elif secim == "4":
                no = input("Silinecek öğrenci no: ").strip()
                sistem.ogrenci_sil(no)

            elif secim == "5":
                ogretmen_id = input("Öğretmen ID: ").strip()
                ad = input("Ad: ").strip()
                soyad = input("Soyad: ").strip()
                brans = input("Branş: ").strip()

                if ogretmen_id and ad and soyad and brans:
                    sistem.ogretmen_ekle(ogretmen_id, ad, soyad, brans)
                else:
                    print("[-] Alanlar boş bırakılamaz.")

            elif secim == "6":
                sistem.ogretmenleri_listele()

            elif secim == "7":
                kod = input("Ders kodu: ").strip().upper()
                ad = input("Ders adı: ").strip()
                ogretmen_id = input("Öğretmen ID: ").strip()

                if kod and ad and ogretmen_id:
                    sistem.ders_ekle(kod, ad, ogretmen_id)
                else:
                    print("[-] Alanlar boş bırakılamaz.")

            elif secim == "8":
                sistem.dersleri_listele()

            elif secim == "9":
                kod = input("Güncellenecek ders kodu: ").strip().upper()
                yeni_ad = input("Yeni ders adı: ").strip()
                yeni_ogretmen_id = input("Yeni öğretmen ID: ").strip()

                if kod and yeni_ad and yeni_ogretmen_id:
                    sistem.ders_guncelle(kod, yeni_ad, yeni_ogretmen_id)
                else:
                    print("[-] Alanlar boş bırakılamaz.")

            elif secim == "10":
                kod = input("Silinecek ders kodu: ").strip().upper()
                sistem.ders_sil(kod)

            elif secim == "11":
                no = input("Öğrenci no: ").strip()
                kod = input("Ders kodu: ").strip().upper()

                try:
                    not_degeri = int(input("Not değeri 0-100: ").strip())
                    sistem.not_gir(no, kod, not_degeri)
                except ValueError:
                    print("[-] Not değeri sayısal olmalıdır.")

            elif secim == "12":
                no = input("Öğrenci no: ").strip()
                sistem.ogrenci_notlarini_goster(no)

            elif secim == "13":
                no = input("Öğrenci no: ").strip()
                sistem.ortalama_hesapla(no)

            elif secim == "14":
                try:
                    baraj = float(input("Başarı barajı, varsayılan 50: ") or 50)
                    sistem.basarili_ogrencileri_listele(baraj)
                except ValueError:
                    print("[-] Geçerli bir sayı giriniz.")

            elif secim == "15":
                arama = input("Öğrenci adı, soyadı veya numarası: ").strip()

                if arama:
                    sistem.ogrenci_ara(arama)
                else:
                    print("[-] Arama metni boş olamaz.")

            elif secim == "16":
                kod = input("Analiz yapılacak ders kodu: ").strip().upper()
                sistem.ders_bazli_basari_analizi(kod)

            elif secim == "17":
                sistem.verileri_kaydet()

            elif secim == "18":
                sistem.verileri_yukle()

            elif secim == "0":
                print("[*] Sistemden çıkılıyor. İyi günler.")
                break

            else:
                print("[-] Geçersiz seçim.")

        except Exception as hata:
            print(f"[-] Beklenmeyen hata oluştu: {hata}")


# Program doğrudan çalıştırıldığında ana menüyü başlatır.
if __name__ == "__main__":
    ana_menu()

[+] Öğretmen başarıyla eklendi.

 ÖĞRENCİ BİLGİ VE NOT TAKİP SİSTEMİ 
1- Öğrenci ekle
2- Öğrencileri listele
3- Öğrenci güncelle
4- Öğrenci sil
5- Öğretmen ekle
6- Öğretmenleri listele
7- Ders ekle
8- Dersleri listele
9- Ders güncelle
10- Ders sil
11- Not gir
12- Öğrenci notlarını görüntüle
13- Ortalama hesapla
14- Başarılı öğrencileri listele
15- Öğrenci ara
16- Ders bazlı başarı analizi
17- Verileri kaydet
18- Verileri yükle
0- Çıkış
[+] Öğrenci başarıyla eklendi.

 ÖĞRENCİ BİLGİ VE NOT TAKİP SİSTEMİ 
1- Öğrenci ekle
2- Öğrencileri listele
3- Öğrenci güncelle
4- Öğrenci sil
5- Öğretmen ekle
6- Öğretmenleri listele
7- Ders ekle
8- Dersleri listele
9- Ders güncelle
10- Ders sil
11- Not gir
12- Öğrenci notlarını görüntüle
13- Ortalama hesapla
14- Başarılı öğrencileri listele
15- Öğrenci ara
16- Ders bazlı başarı analizi
17- Verileri kaydet
18- Verileri yükle
0- Çıkış
[-] Ders bulunamadı.

 ÖĞRENCİ BİLGİ VE NOT TAKİP SİSTEMİ 
1- Öğrenci ekle
2- Öğrencileri listele
3- Öğrenci güncelle
4- 